In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from collections import Counter
from itertools import combinations
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
warnings.filterwarnings("ignore")

In [19]:
# Load data
df = pd.read_csv("train.csv")

In [20]:
df["capturedAt"] = pd.to_datetime(df["capturedAt"])

### Drop duplicates and unnecessary fields

In [21]:
df = df.drop_duplicates(keep="first")

In [22]:
df = df.drop(
    columns=["stock", "normal_stock", "is_preferred_plus_seller", "show_discount"]
)

### Fill Missing Values

In [23]:
# Fill brand
brand_list = df["brand"].value_counts().index.tolist()
UNKNOWN_BRAND_TAG = "unknown"
if not UNKNOWN_BRAND_TAG in [b.lower() for b in brand_list if isinstance(b, str)]:
    df["brand"] = df["brand"].fillna(UNKNOWN_BRAND_TAG)

In [24]:
# normalize price
df["price"] = df["price"] / 100_000
df["item_price_min"] = df["item_price_min"] / 100_000
df["item_price_max"] = df["item_price_max"] / 100_000
df["priceBeforeDiscount"] = df["priceBeforeDiscount"] / 100_000

In [25]:
def fill_shop_response_rate(df):
    df = df.copy()
    df = df.sort_values(["shopId", "capturedAt"])
    def fill_group(group):
        group = group.copy()
        group["shop_response_rate"] = group["shop_response_rate"].interpolate(
            method="linear"
        )
        group["shop_response_rate"] = group["shop_response_rate"].bfill()
        group["shop_response_rate"] = group["shop_response_rate"].ffill()

        return group
    return (
        df.groupby("shopId", group_keys=False)
          .apply(fill_group)
          .reset_index(drop=True)
    )

In [26]:
df = fill_shop_response_rate(df)

In [27]:
df.isnull().sum()

capturedAt             0
shopId                 0
itemId                 0
modelId                0
price                  0
priceBeforeDiscount    0
promotionId            0
cat_id                 0
raw_discount           0
brand                  0
is_free_shipping       0
is_pre_order           0
item_price_min         0
item_price_max         0
review_rating          0
total_rating_count     0
cmt_count              0
shop_rating            0
shop_response_rate     0
shop_follower_count    0
is_official_shop       0
is_verified            0
dtype: int64

In [28]:
len(df)

303982

### Boolean Encoding

In [29]:
is_cols = [col for col in df.columns if col.startswith("is_")]
is_cols

['is_free_shipping', 'is_pre_order', 'is_official_shop', 'is_verified']

In [30]:
df[is_cols] = (
    df[is_cols]
    .replace({"t": 1, "f": 0})
    .astype("Int64")
)

### Remove Inconsistent Records

In [31]:
def remove_promotion_without_discount(df):
    mask = (
        (df["promotionId"] != 0) &
        (df["raw_discount"] == 0)
    )

    removed = mask.sum()
    return df.loc[~mask].copy(), removed


def remove_discount_without_original_price(df):
    mask = (
        (df["priceBeforeDiscount"] == 0) &
        (df["raw_discount"] != 0)
    )

    removed = mask.sum()
    return df.loc[~mask].copy(), removed


def remove_price_mismatch(df, threshold=60):
    valid_mask = (
        (df["priceBeforeDiscount"] != 0) &
        (df["raw_discount"] != 0)
    )

    expected_price = (
        df.loc[valid_mask, "priceBeforeDiscount"] *
        (1 - df.loc[valid_mask, "raw_discount"] / 100)
    )

    diff_price = (
        df.loc[valid_mask, "price"] - expected_price
    ).abs()

    diff_price_pct = (
        diff_price /
        df.loc[valid_mask, "priceBeforeDiscount"] * 100
    )

    mismatch_mask = pd.Series(False, index=df.index)
    mismatch_mask.loc[valid_mask] = diff_price_pct > threshold

    removed = mismatch_mask.sum()
    return df.loc[~mismatch_mask].copy(), removed

In [32]:
def remove_price_outside_item_range(df, threshold=-0.6):
    diff_min_pct = (
        (df["price"] - df["item_price_min"]) /
        df["price"]
    )

    diff_max_pct = (
        (df["item_price_max"] - df["price"]) /
        df["price"]
    )

    mask = (
        (diff_min_pct < threshold) |
        (diff_max_pct < threshold)
    )

    removed = mask.sum()
    return df.loc[~mask].copy(), removed

In [33]:
def remove_price_outside_item_iqr(df):
    item_counts = (
        df.groupby("itemId")["price"]
          .nunique()
    )

    multiple_price_items = item_counts[item_counts > 1].index
    stats = (
        df[df["itemId"].isin(multiple_price_items)]
        .groupby("itemId")["price"]
        .agg(
            q1=lambda x: x.quantile(0.25),
            q3=lambda x: x.quantile(0.75),
        )
        .reset_index()
    )

    stats["iqr"] = stats["q3"] - stats["q1"]
    stats["lower_bound"] = stats["q1"] - 1.5 * stats["iqr"]
    stats["upper_bound"] = stats["q3"] + 1.5 * stats["iqr"]

    df_check = df.merge(
        stats[["itemId", "lower_bound", "upper_bound"]],
        on="itemId",
        how="left"
    )

    mask = (
        df_check["lower_bound"].notna() &
        (
            (df_check["price"] < df_check["lower_bound"]) |
            (df_check["price"] > df_check["upper_bound"])
        )
    )

    removed = mask.sum()

    return (
        df_check.loc[~mask]
            .drop(columns=["lower_bound", "upper_bound"]),
        removed
    )

In [34]:
df_clean = df.copy()

rows_before = len(df_clean)
price_mismatch_threshold = 60
item_range_threshold = -0.6

df_clean, promotion_removed = remove_promotion_without_discount(df_clean)
df_clean, discount_removed = remove_discount_without_original_price(df_clean)
df_clean, price_mismatch_removed = remove_price_mismatch(
    df_clean,
    threshold=price_mismatch_threshold
)
df_clean, item_range_removed = remove_price_outside_item_range(
    df_clean,
    threshold=item_range_threshold
)
df_clean, item_iqr_removed = remove_price_outside_item_iqr(df_clean)

rows_removed = rows_before - len(df_clean)

print(
    f"[1] Removed {promotion_removed:,} records with promotion but no discount "
    f"(promotionId != 0 and raw_discount == 0)."
)

print(
    f"[2] Removed {discount_removed:,} records with discount but no original price "
    f"(priceBeforeDiscount == 0 and raw_discount != 0)."
)

print(
    f"[3] Removed {price_mismatch_removed:,} records with large discounted price mismatch "
    f"(price differs by more than {price_mismatch_threshold}% from the expected discounted price)."
)

print(
    f"[4] Removed {item_range_removed:,} records with prices far below the historical item price range "
    f"(relative difference from the item's historical minimum or maximum price < {item_range_threshold}, "
    f"i.e., more than {abs(item_range_threshold):.0%} below the historical boundary)."
)
print(
    f"[5] Removed {item_iqr_removed:,} records with prices outside the IQR range "
    f"of their itemId."
)

print(f"[6] Total removed: {rows_removed:,}")
print(f"[7] Remaining rows: {len(df_clean):,}")

[1] Removed 3 records with promotion but no discount (promotionId != 0 and raw_discount == 0).
[2] Removed 8,465 records with discount but no original price (priceBeforeDiscount == 0 and raw_discount != 0).
[3] Removed 920 records with large discounted price mismatch (price differs by more than 60% from the expected discounted price).
[4] Removed 760 records with prices far below the historical item price range (relative difference from the item's historical minimum or maximum price < -0.6, i.e., more than 60% below the historical boundary).
[5] Removed 12,687 records with prices outside the IQR range of their itemId.
[6] Total removed: 22,835
[7] Remaining rows: 281,147


In [35]:
len(df), len(df_clean)

(303982, 281147)

In [36]:
df_clean.to_csv("train_clean.csv", index=False)

In [37]:
df.dtypes

capturedAt             datetime64[ns]
shopId                          int64
itemId                          int64
modelId                         int64
price                         float64
priceBeforeDiscount           float64
promotionId                     int64
cat_id                          int64
raw_discount                    int64
brand                          object
is_free_shipping                Int64
is_pre_order                    Int64
item_price_min                float64
item_price_max                float64
review_rating                 float64
total_rating_count              int64
cmt_count                       int64
shop_rating                   float64
shop_response_rate            float64
shop_follower_count             int64
is_official_shop                Int64
is_verified                     Int64
dtype: object

In [38]:
df_clean.dtypes

capturedAt             datetime64[ns]
shopId                          int64
itemId                          int64
modelId                         int64
price                         float64
priceBeforeDiscount           float64
promotionId                     int64
cat_id                          int64
raw_discount                    int64
brand                          object
is_free_shipping                Int64
is_pre_order                    Int64
item_price_min                float64
item_price_max                float64
review_rating                 float64
total_rating_count              int64
cmt_count                       int64
shop_rating                   float64
shop_response_rate            float64
shop_follower_count             int64
is_official_shop                Int64
is_verified                     Int64
dtype: object